In [ ]:
# Packages
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Download latest version.
path = kagglehub.dataset_download("danieltreiman/twitter-human-bots-dataset", output_dir="data/twitter-human-bots-dataset")
print("Path to dataset files:", path + "/" )

In [ ]:
# Reads bot_detection_data.csv in the form of pandas data frame.
df = pd.read_csv("data/twitter-human-bots-dataset/twitter_human_bots_dataset.csv")
print(df.shape)

In [ ]:
# Dropping all non-useful features. I am doing this for now, but we might have lost some valuable data doing this.
# Figuring out how we can use these these features can be a future improvement.
# The features that would be cool to include in the machine learning algorithm would be the images and the texts.
# For images we can use some kind of pretrained image neural network, for words we can use entropy.
columns_to_drop = [
    'Unnamed: 0',
    'created_at',
    'id',
    'lang',
    'location',
    'profile_background_image_url',
    'profile_image_url',
    'screen_name',
    'profile_image_path',
    'profile_background_image_path',
    'split',
]

df = df.drop(columns=columns_to_drop)

# Convert boolean attributes to numeric values (0 and 1).
boolean_columns = [
    'default_profile',
    'default_profile_image',
    'geo_enabled',
    'verified'
]

df[boolean_columns] = df[boolean_columns].astype(int)

# Binary flag: 0 = empty, 1 = has a description
df['description'] = df['description'].notna().astype(int)

# Encode account type as a binary label (1 = bot, 0 = human).
df['account_type'] = (df['account_type'] == 'bot').astype(int)

# Dropping all tuples that contain empty or null values. I am doing this for now, but we might have lost some valuable data doing this.
# Figuring out a better way to do this can be a future improvement.
df = df.dropna()
print(df.shape)

df.to_csv("data/twitter-human-bots-dataset/twitter-human-bots-dataset_processed.csv", index=False)

In [ ]:
y = df['account_type']
x = df.drop(columns=['account_type'])

# List of features for later use
feature_list = list(x.columns)
print(feature_list)

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.3,
    random_state=42
)

In [ ]:
# Random forest.
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    oob_score=True,
    criterion='entropy',
)

rf_model.fit(x_train, y_train)

In [ ]:
# Random forest results.
rf_y_pred = rf_model.predict(x_test)

print(classification_report(y_test, rf_y_pred))
print("Accuracy_score:", accuracy_score(y_test, rf_y_pred))

In [ ]:
# Get the importance score of each feature from the trained model.
importances = rf_model.feature_importances_

# Pair each feature name with its importance score.
feature_importance_pairs = list(zip(feature_list, importances))

# Sort from most to least important.
feature_importance_pairs.sort(key=lambda pair: pair[1], reverse=True)

# Print each feature and its importance, rounded to 2 decimal places
for feature, importance in feature_importance_pairs:
    print(f"{feature:<20} Importance: {importance:.2f}")

In [ ]:
# List of features sorted from most to least important
sorted_importances = [importance[1] for importance in feature_importances]
sorted_features = [importance[0] for importance in feature_importances]
# Cumulative importances
cumulative_importances = np.cumsum(sorted_importances)
# Make a line graph
plt.plot(x_values, cumulative_importances, 'g-')
# Draw line at 95% of importance retained
plt.hlines(y = 0.95, xmin=0, xmax=len(sorted_importances), color = 'r', linestyles = 'dashed')
# Format x ticks and labels
plt.xticks(x_values, sorted_features, rotation = 'vertical')
# Axis labels and title
plt.xlabel('Variable'); plt.ylabel('Cumulative Importance'); plt.title('Cumulative Importances');

In [ ]:
low_importance_features = [
    'default_profile',
    'geo_enabled',
    'description',
    'default_profile_image'
]

df = df.drop(columns=low_importance_features)

y_2 = df['account_type']
x_2 = df.drop(columns=['account_type'])

# List of features for later use
feature_list = list(x_2.columns)
print(feature_list)

x_train_2, x_test_2, y_train_2, y_test_2 = train_test_split(
    x_2, y_2,
    test_size=0.3,
    random_state=42
)

# Random forest.
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    oob_score=True,
    criterion='entropy',
)

rf_model.fit(x_train_2, y_train_2)


# Random forest results.
rf_y_pred_2 = rf_model.predict(x_test_2)

print(classification_report(y_test_2, rf_y_pred_2))
print("Accuracy_score:", accuracy_score(y_test_2, rf_y_pred_2))